# OneVoice V2 — Fine-tune SenseVoice cho English construction ASR

Notebook này chỉ fine-tune ASR English. MT VI→EN và EN→VI đã được đóng ở candidate V1 nên không được train lại ở đây. Cần chọn **GPU runtime**. Dữ liệu và checkpoint đều ở Google Drive; chạy lại sau khi Colab ngắt sẽ để FunASR resume từ output hiện có.

Training dùng clean + noisy của `train`, validation dùng clean + noisy của `dev`; `test` không được đưa vào train/validation.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import json, os, subprocess, sys

GITHUB_REPO = 'https://github.com/Platypus27-coder/OneVoice.git'
BRANCH = 'main'
REPO = Path('/content/OneVoice')
MYDRIVE = Path('/content/drive/MyDrive')
WORK_ROOT = MYDRIVE / 'OneVoice'
DATASET_ROOT = MYDRIVE / 'onevoice_audio_v2_1'
MANIFEST = DATASET_ROOT / 'manifest.jsonl'
PREPARED_ROOT = WORK_ROOT / 'datasets/sensevoice_en_construction_v1'
MODEL_ROOT = WORK_ROOT / 'models/sensevoice_en_construction_v1'

if (REPO / '.git').is_dir():
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, GITHUB_REPO, str(REPO)], check=True)
os.chdir(REPO)
os.environ['PYTHONUNBUFFERED'] = '1'
os.environ['MODELSCOPE_CACHE'] = str(WORK_ROOT / 'model_cache/modelscope')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'numpy==2.2.6', 'funasr>=1.4.3', 'modelscope', 'transformers==4.57.1', 'tokenizers==0.22.1', 'sentencepiece==0.2.0', 'accelerate', 'soundfile'], check=True)

import torch
if not torch.cuda.is_available():
    raise RuntimeError('Chọn Runtime → Change runtime type → GPU, rồi chạy lại từ cell này. Fine-tune SenseVoice không chạy CPU.')
if not MANIFEST.is_file():
    raise FileNotFoundError(f'Không tìm thấy English V2.1 manifest: {MANIFEST}')
print('GPU:', torch.cuda.get_device_name(0))
print('Source:', REPO)
print('Data:', MANIFEST)
print('Prepared data:', PREPARED_ROOT)
print('Checkpoint/log:', MODEL_ROOT)


In [ ]:
# Chuẩn bị JSONL ChatML đúng format FunASR. Không quét lại 24.192 WAV vì audit vật lý đã hoàn tất.
required = [PREPARED_ROOT / name for name in ('train.jsonl', 'dev.jsonl', 'dataset_manifest.json')]
if not all(path.is_file() for path in required):
    subprocess.run([sys.executable, 'scripts/prepare_sensevoice_finetune_data.py', str(MANIFEST), '--output-dir', str(PREPARED_ROOT), '--text-length-mode', 'qwen'], check=True)
else:
    print('Prepared FunASR data already exists on Drive; keeping it for reproducibility/resume.')

prepared = json.loads((PREPARED_ROOT / 'dataset_manifest.json').read_text(encoding='utf-8'))
assert prepared['test_split_included'] is False
display(prepared)


In [ ]:
# Official FunASR/SenseVoice trainer. Adapter + LLM train; audio encoder stays frozen for this small construction corpus.
import funasr
funasr_root = Path(funasr.__file__).resolve().parent
candidates = [funasr_root / 'bin/train_ds.py', *Path(sys.prefix).glob('lib/python*/site-packages/funasr/bin/train_ds.py')]
TRAIN_TOOL = next((path for path in candidates if path.is_file()), None)
if TRAIN_TOOL is None:
    raise FileNotFoundError('FunASR train_ds.py not found after installation.')

MODEL_ROOT.mkdir(parents=True, exist_ok=True)
command = [
    'torchrun', '--standalone', '--nproc_per_node=1', str(TRAIN_TOOL),
    '++model=iic/SenseVoiceSmall',
    '++trust_remote_code=true',
    f'++train_data_set_list={PREPARED_ROOT / "train.jsonl"}',
    f'++valid_data_set_list={PREPARED_ROOT / "dev.jsonl"}',
    '++dataset_conf.data_split_num=1',
    '++dataset_conf.batch_sampler=BatchSampler',
    '++dataset_conf.batch_size=2400',
    '++dataset_conf.sort_size=512',
    '++dataset_conf.batch_type=token',
    '++dataset_conf.num_workers=2',
    '++train_conf.max_epoch=10',
    '++train_conf.log_interval=20',
    '++train_conf.resume=true',
    '++train_conf.validate_interval=500',
    '++train_conf.save_checkpoint_interval=500',
    '++train_conf.keep_nbest_models=3',
    '++train_conf.avg_nbest_model=3',
    '++train_conf.use_deepspeed=false',
    '++optim_conf.lr=0.0001',
    '++model.audio_encoder_conf.freeze=true',
    '++model.audio_adaptor_conf.freeze=false',
    '++model.llm_conf.freeze=false',
    f'++output_dir={MODEL_ROOT}',
]
print('> ' + ' '.join(map(str, command)), flush=True)
log_path = MODEL_ROOT / 'training.log'
with log_path.open('a', encoding='utf-8') as log:
    log.write('\n\n=== resumed launcher ===\n' + ' '.join(map(str, command)) + '\n')
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env={**os.environ, 'PYTHONUNBUFFERED': '1'})
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='', flush=True)
        log.write(line)
        log.flush()
    code = process.wait()
if code:
    raise RuntimeError(f'Fine-tune stopped with exit code {code}. Check {log_path}; keep the same MODEL_ROOT and re-run this cell to resume.')
print(f'Fine-tune command completed. Persistent log: {log_path}')


In [ ]:
# Chỉ hiển thị checkpoint trên Drive; chưa thay runtime ONNX bằng checkpoint này.
files = sorted(path.relative_to(MODEL_ROOT).as_posix() for path in MODEL_ROOT.rglob('*') if path.is_file())
print('Files:', len(files))
print('\n'.join(files[-50:]))
print('\nBước tiếp theo sau khi hoàn tất: export/đánh giá checkpoint trên dev/test trong notebook ASR riêng; không dùng test để train.')
